# Library Import

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, HTML

from sklearn.model_selection import train_test_split

import torch
from torch import nn
from torch.autograd import Variable
from torch.utils.data import Dataset, DataLoader, TensorDataset

import warnings
warnings.filterwarnings('ignore')


# Loaded the Dataset

In [ ]:
data = pd.read_csv('/kaggle/input/digit-recognizer/train.csv', dtype=np.float32)

data

# Split the dataset into Feature and labels

In [ ]:
y = data.label.values

X = data.drop(columns=['label']).values/255  # normalize

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)
print(f'X_train: {X_train.shape}')
print(f'X_test {X_test.shape}')

In [ ]:
X_train_tensor = torch.from_numpy(X_train)
y_train_tensor = torch.from_numpy(y_train).type(torch.LongTensor)  # data type is long

X_test_tensor = torch.from_numpy(X_test)
y_test_tensor = torch.from_numpy(y_test).type(torch.LongTensor)  # data type is long

# create PyTorch dataset
train_data = TensorDataset(X_train_tensor, y_train_tensor)
test_data = TensorDataset(X_test_tensor, y_test_tensor)

In [ ]:
batch_size = 96
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32)

# Visualize some Dataset

In [ ]:
_, ax = plt.subplots(3, 8, figsize=(16, 6)) 
torch.manual_seed(81)
visualize_loader = DataLoader(train_data, batch_size=1, shuffle=True)

for i, (data, label) in enumerate(visualize_loader):
    data = data.reshape((-1, 28, 28))[:,:,:,None]
    ax[i//8][i%8].imshow(data.cpu().squeeze())
    ax[i//8][i%8].axis('off')
    ax[i//8][i%8].set_title(f"Label: {label.item()}", y=-0.3)
    if(i==23): 
        break



plt.subplots_adjust(hspace=0.5)
plt.savefig('plot2.png')
plt.close()

# Display the saved plot file in HTML
display(HTML("<center><img src='plot2.png'></center>"))

# RNN model define

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Hyperparameters
input_dim = 28
hidden_dim = 200
layer_dim = 1
output_dim = 10
learning_rate = 0.002


class RNNmodel(nn.Module):
    def __init__(self, device):
        super(RNNmodel,self).__init__()
        self.device = device
        self.rnn = nn.RNN(input_dim, hidden_dim, layer_dim, batch_first=True, nonlinearity='relu')
        self.fc = nn.Linear(hidden_dim,output_dim)

    def forward(self,x):
        h0 = torch.zeros(layer_dim, x.size(0),hidden_dim).to(self.device)
        out,_ = self.rnn(x, h0)
        out = self.fc(out[:,-1,:])

        return out

    

In [ ]:
model = RNNmodel(device).to(device)

criterion = nn.CrossEntropyLoss().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=0.001)

# Training RNN model

In [ ]:
seq_dim = 28  
num_epochs=50
metrics = {'loss': [], 'iteration': [], 'accuracy': []}
count = 0

for epoch in range(num_epochs):

    for i, (image, label) in enumerate(train_loader):
        train = Variable(image.view(-1,seq_dim,input_dim)).to(device)
        label = Variable(label).to(device)
        image = image.to(device)

        optimizer.zero_grad()
        output = model(train)
        loss = criterion(output, label)
        loss.backward()
        optimizer.step()

        count +=1
        if count % 250 == 0:
            correct, total =0,0
            for img, labels in test_loader:
                img = Variable(img.view(-1, seq_dim, input_dim)).to(device)

                labels = labels.to(device)
                out = model(img).to(device)

                predicted = torch.max(out.data,1)[1]
                total += labels.size(0)
                correct += (predicted == labels).sum()

            accuracy = 100 * correct / float(total)
            metrics['loss'].append(loss.item())
            metrics['iteration'].append(count)
            metrics['accuracy'].append(accuracy.item())

            if count % 500 == 0:
                print(f'Iteration: {count}  Loss: {loss}  Accuracy: {accuracy} %')
        

In [ ]:
# save the final model
torch.save(model.state_dict(),"RNN_model.pth")

# Visualized training curve

In [ ]:
import seaborn as sns

# Use seaborn style defaults and set the default figure size
sns.set(rc={'figure.figsize':(16, 8)})

fig, axs = plt.subplots(ncols=2)

# Loss plot
sns.lineplot(x=metrics['iteration'], y=metrics['loss'], ax=axs[0], linewidth=2.5)
axs[0].set_title('RNN: Loss vs Number of iteration', fontsize=14, fontweight='bold')
axs[0].set_xlabel('Number of iteration', fontsize=12)
axs[0].set_ylabel('Loss', fontsize=12)
axs[0].grid(True)

# Accuracy plot
sns.lineplot(x=metrics['iteration'], y=metrics['accuracy'], ax=axs[1], color="red", linewidth=2.5)
axs[1].set_title('RNN: Accuracy vs Number of iteration', fontsize=14, fontweight='bold')
axs[1].set_xlabel('Number of iteration', fontsize=12)
axs[1].set_ylabel('Accuracy', fontsize=12)
axs[1].grid(True)

plt.tight_layout()
plt.savefig('graph.png')
plt.close()
#plt.show()
display(HTML("<center><img src='graph.png'></center>"))

# visualize the some predicted image

In [ ]:
# quick debug to see shapes from your dataloader
batch = next(iter(test_loader))
images, labels = batch
print("images.shape =", images.shape)   # expect (32, 784)
print("labels.shape =", labels.shape)   # expect (32,) or (32,1)


In [ ]:
sns.reset_orig()

# ---------------- Config ----------------
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

cols = 8
rows = 4   # 4*8 = 32 images to show
fig, axes = plt.subplots(rows, cols, figsize=(20, 10))
fig.subplots_adjust(hspace=0.5, wspace=0.4)

data_iter = iter(test_loader)
shown = 0
max_show = rows * cols  # 32

# helper: convert 784-vector(s) to (H,W) or (B,H,W)
def vec_to_image(arr, H=28, W=28):
    arr = np.array(arr)
    if arr.ndim == 1:
        return arr.reshape(H, W)
    elif arr.ndim == 2:
        return arr.reshape(arr.shape[0], H, W)
    else:
        raise ValueError("Expected 1D or 2D array, got: " + str(arr.shape))


with torch.no_grad():
    while shown < max_show:
        try:
            images, labels = next(data_iter)  # images: (B,784), labels: (B,)
        except StopIteration:
            break

        # safety: ensure shapes
        if images.dim() == 1:
            images = images.unsqueeze(0)
        if labels.dim() == 2 and labels.size(1) == 1:
            labels = labels.squeeze(1)

        B = images.size(0)

        # Move to device (and ensure float)
        images_device = images.to(device, dtype=torch.float32)  # (B, 784)
        labels_device = labels.to(device)

        # uncomment and set seq_dim, input_dim accordingly (e.g., 28,28):
        seq_dim = 28
        input_dim = 28
        outputs = model(images_device.view(-1, seq_dim, input_dim))

        # If model returns a ModelOutput with .logits (HuggingFace), extract it:
        if hasattr(outputs, "logits"):
            logits = outputs.logits
        else:
            logits = outputs

        # Get predictions safely for the batch
        if logits.dim() == 2:             # typical classifier: [B, num_classes]
            preds = logits.argmax(dim=1)  # shape [B]
        elif logits.dim() == 1:           # maybe already indices
            preds = logits
        else:
            raise RuntimeError(f"Unexpected logits shape: {logits.shape}")

        # Convert images to numpy and reshape for visualization
        imgs_np = images.cpu().numpy()         # shape (B, 784)
        imgs_reshaped = vec_to_image(imgs_np)  # shape (B, 28, 28)

        # plot each image in the batch (or until we reach max_show)
        for b in range(B):
            if shown >= max_show:
                break
            ax = axes[shown // cols][shown % cols]

            img = imgs_reshaped[b]
            ax.imshow(img)
            ax.axis('off')

            pred_label = int(preds[b].cpu().item())
            true_label = int(labels_device[b].cpu().item())

            title = f"Pred: {pred_label}\nTrue: {true_label}"
            color = 'green' if pred_label == true_label else 'red'
            ax.set_title(title, color=color, y=-0.5)

            shown += 1

# Hide any unused axes
total_axes = rows * cols
for idx in range(shown, total_axes):
    axes[idx // cols][idx % cols].axis('off')

plt.savefig('plot.png', bbox_inches='tight')
plt.close()
display(HTML("<center><img src='plot.png'></center>"))

In [ ]:
test = pd.read_csv('/kaggle/input/digit-recognizer/test.csv', dtype=np.float32)

test

# Created Submission file

In [ ]:
path ='/kaggle/working/RNN_model.pth'
model.load_state_dict(torch.load(path, map_location=device))

model.to(device)
model.eval()


pixel_cols = [c for c in test.columns if c.startswith('pixel')]
X = test[pixel_cols].values.astype(np.float32)
tensor_x = torch.from_numpy(X)
test_dataset = TensorDataset(tensor_x)   # no labels
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

preds_all = []
with torch.no_grad():
    for (batch_x,) in test_loader:
        batch_x = batch_x.to(device, dtype=torch.float32)
        # reshape if model expects (B, seq_dim, input_dim)
        batch_in = batch_x.view(batch_x.size(0), seq_dim, input_dim)
        outputs = model(batch_in)
        # outputs shape expected [B, num_classes]
        if hasattr(outputs, "logits"):
            logits = outputs.logits
        else:
            logits = outputs
        batch_preds = logits.argmax(dim=1).cpu().numpy().astype(int)
        preds_all.extend(batch_preds.tolist())


# build submission DataFrame
image_ids = np.arange(1, len(preds_all) + 1)
submission_df = pd.DataFrame({'ImageId': image_ids, 'Label': preds_all})
submission_df.to_csv("submission.csv", index=False)
print("Saved submission.csv with", len(preds_all), "rows")

In [ ]:
submission_df